# PlantSeg Stage 2 — Session 3 of 3
## Aggregator (binary lesion segmentation)

This is the **Aggregator session**, run after Session 1 and Session 2 have each trained 3 models and you've collected all 6 `s2_result_*.pkl` files into one Kaggle Dataset. This notebook is reproduced **exactly as-is** from the original `PlantSeg_S2_aggregate.ipynb` — no logic changes. No retraining; CPU-only.

### ⚠️ Required setup before running
Attach **two** Kaggle input sources:
1. Your combined results dataset containing all 6 `s2_result_*.pkl` files (from Session 1 + Session 2). If named `s2-results`, it mounts at `/kaggle/input/s2-results/` — the default `RESULTS_IN` path below. If named differently, edit `RESULTS_IN` in the path-setup cell.
2. The **PlantSeg dataset** (for the dataset figures: sample images + GT lesion masks, lesion-area distribution).

**Builds:** comparison table (mIoU / fg-IoU / Dice / pixel-acc), training-curve grid, accuracy–efficiency frontier, bootstrap 95% CI on per-image fg-IoU, and dataset figures.

In [ ]:
import os, json, pickle, math
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from PIL import Image, ImageDraw
from collections import defaultdict
sns.set_theme(style="whitegrid", context="talk")

**Paths** — locate the 6 result pickles and the PlantSeg data root (with Kaggle/local fallbacks):

In [ ]:
RESULTS_IN=Path("/kaggle/input/s2-results")
if not list(RESULTS_IN.glob("s2_result_*.pkl")):
    for c in [Path("/kaggle/working/s2_results"),Path("s2_results"),Path(".")]:
        if list(c.glob("s2_result_*.pkl")): RESULTS_IN=c; break
DATA_ROOT=Path("/kaggle/input/plantseg/plantseg")
if not (DATA_ROOT/"Metadata.csv").exists():
    for c in [Path("/kaggle/input/plantseg"),Path("plantseg"),Path(".")]:
        if (c/"Metadata.csv").exists(): DATA_ROOT=c; break
OUT=Path("/kaggle/working/s2_aggregate"); FIG=OUT/"figures"; RES=OUT/"results"
FIG.mkdir(parents=True,exist_ok=True); RES.mkdir(parents=True,exist_ok=True)
print("results:", sorted(p.name for p in RESULTS_IN.glob('s2_result_*.pkl')))

**Load results** — reads each `s2_result_*.pkl`, builds the metrics comparison dataframe (sorted by fg-IoU), identifies the best model:

In [ ]:
# load results
rows={}; hist={}; pifg={}; dep={}
for p in sorted(RESULTS_IN.glob("s2_result_*.pkl")):
    r=pickle.load(open(p,"rb")); n=r["model_name"]
    rows[n]={**r["test_scores"],**r["deployment"],"train_time_s":r["train_time_sec"]}
    hist[n]=r["history"]; pifg[n]=np.array(r["per_image_fg_iou"]); dep[n]=r["deployment"]
metrics_df=pd.DataFrame(rows).T.sort_values("fg_IoU",ascending=False)
metrics_df.to_csv(RES/"s2_model_comparison.csv"); print(metrics_df.round(4))
best=metrics_df.index[0]; print("BEST (fg-IoU):",best)

**Rendered comparison table** — saves a formatted table image (mIoU / fg-IoU / mDice / fg-Dice / pixel-acc / params / GPU latency):

In [ ]:
# rendered comparison table
cols=["mIoU","fg_IoU","mDice","fg_Dice","pixel_acc","params_M","latency_gpu_ms"]
cols=[c for c in cols if c in metrics_df.columns]; tbl=metrics_df[cols].round(4)
fig,ax=plt.subplots(figsize=(2+1.7*len(cols),0.6*len(tbl)+1.2)); ax.axis("off")
t=ax.table(cellText=tbl.values,rowLabels=tbl.index,colLabels=cols,loc="center",cellLoc="center")
t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1,1.5)
for j in range(len(cols)): t[0,j].set_facecolor("#4C72B0"); t[0,j].set_text_props(color="w",weight="bold")
plt.title("Binary lesion segmentation — model comparison (sorted by fg-IoU)",pad=12)
plt.tight_layout(); plt.savefig(FIG/"s2_table_comparison.png",dpi=200,bbox_inches="tight"); plt.show()

**Training curves + frontier + metric bars** — grid of per-model training curves, grouped metric bar chart, and the accuracy–efficiency (latency vs. fg-IoU) scatter:

In [ ]:
# training curves + frontier + metric bars
n=len(hist); cols=2; rows=math.ceil(n/cols); fig,axes=plt.subplots(rows,cols,figsize=(16,4*rows)); axes=np.array(axes).flatten()
for ax,(nm,h) in zip(axes,hist.items()):
    ep=range(1,len(h["train_loss"])+1); ax.plot(ep,h["train_loss"],"--",label="loss")
    a2=ax.twinx(); a2.plot(ep,h["val_mIoU"],color="green"); a2.plot(ep,h["val_fgIoU"],color="orange"); ax.set_title(nm); ax.set_xlabel("epoch")
for ax in axes[n:]: ax.axis("off")
plt.tight_layout(); plt.savefig(FIG/"s2_training_curves.png",dpi=150,bbox_inches="tight"); plt.show()

ax=metrics_df[["mIoU","fg_IoU","mDice"]].plot(kind="bar",figsize=(13,7)); ax.set_title("Segmentation metrics by model"); ax.set_ylabel("score")
plt.xticks(rotation=25,ha="right"); plt.tight_layout(); plt.savefig(FIG/"s2_metric_bars.png",dpi=200,bbox_inches="tight"); plt.show()

fig,ax=plt.subplots(figsize=(11,7))
for nm in metrics_df.index:
    ax.scatter(metrics_df.loc[nm,"latency_gpu_ms"],metrics_df.loc[nm,"fg_IoU"],s=120)
    ax.annotate(nm,(metrics_df.loc[nm,"latency_gpu_ms"],metrics_df.loc[nm,"fg_IoU"]),fontsize=8,xytext=(4,4),textcoords="offset points")
ax.set_xlabel("GPU latency (ms)"); ax.set_ylabel("fg-IoU"); ax.set_title("Accuracy–efficiency frontier")
plt.tight_layout(); plt.savefig(FIG/"s2_frontier.png",dpi=200,bbox_inches="tight"); plt.show()

**Bootstrap 95% CI** — 1000-resample bootstrap on the best model's per-image fg-IoU distribution:

In [ ]:
# bootstrap 95% CI on per-image fg-IoU for the best model
fg=pifg[best]; rng=np.random.RandomState(42); boot=[rng.choice(fg,len(fg),replace=True).mean() for _ in range(1000)]
ci={"best_model":best,"fg_iou_mean":float(fg.mean()),"ci95":[float(np.percentile(boot,2.5)),float(np.percentile(boot,97.5))]}
json.dump(ci,open(RES/"s2_stats.json","w"),indent=2); print(ci)

**Dataset figures** — sample images with ground-truth lesion mask overlays (red), and the lesion-area (mask ratio) histogram; writes the final `s2_summary.json`:

In [ ]:
# dataset figures: sample images + GT lesion masks, lesion-area distribution
meta=pd.read_csv(DATA_ROOT/"Metadata.csv",encoding="utf-8-sig")
meta["split"]=meta["Split"].map({"Training":"train","Validation":"val","Test":"test"})
meta["filepath"]=meta.apply(lambda r: DATA_ROOT/"images"/r["split"]/r["Name"],axis=1)
def load_anns(split):
    d=json.load(open(DATA_ROOT/f"annotation_{split}.json")); by=defaultdict(list)
    for a in d["annotations"]: by[a["image_id"]].append(a)
    return {im["file_name"]:(im["width"],im["height"],[s for a in by.get(im["id"],[]) for s in a["segmentation"] if len(s)>=6]) for im in d["images"]}
fn=load_anns("train")
def overlay(p,polys,wh,alpha=0.45):
    im=Image.open(p).convert("RGB"); m=Image.new("L",wh,0); dr=ImageDraw.Draw(m)
    for s in polys: dr.polygon(list(zip(s[0::2],s[1::2])),fill=1)
    a=np.asarray(im).astype(float)/255; mk=np.asarray(m)>0; a[mk]=(1-alpha)*a[mk]+alpha*np.array([1,0,0]); return a
rows_=meta[meta.split=="train"].groupby("Plant").first().reset_index().head(8)
fig,axes=plt.subplots(2,4,figsize=(16,8))
for ax,(_,r) in zip(axes.flatten(),rows_.iterrows()):
    w,h,polys=fn.get(r["Name"],(None,None,[]))
    if w: ax.imshow(overlay(r.filepath,polys,(w,h)))
    ax.set_title(f"{r['Plant']}",fontsize=9); ax.axis("off")
plt.suptitle("Ground-truth lesion masks (red overlay)"); plt.tight_layout(); plt.savefig(FIG/"s2_sample_masks.png",dpi=150,bbox_inches="tight"); plt.show()

if "Mask ratio" in meta.columns:
    plt.figure(figsize=(9,5)); plt.hist(meta["Mask ratio"].dropna(),bins=50,color="#4C72B0")
    plt.title("Lesion-area ratio (mask/image)"); plt.xlabel("ratio"); plt.ylabel("images")
    plt.tight_layout(); plt.savefig(FIG/"s2_lesion_area_dist.png",dpi=200,bbox_inches="tight"); plt.show()
json.dump({"best":best,"metrics":metrics_df.round(5).to_dict(),"stats":ci},open(RES/"s2_summary.json","w"),indent=2,default=str)
print("saved aggregate to",RES)

---
## End of Pipeline

Final outputs are in `/kaggle/working/s2_aggregate/`:
- `results/s2_model_comparison.csv` — full metrics table
- `results/s2_stats.json` — bootstrap CI for the best model
- `results/s2_summary.json` — everything combined
- `figures/` — comparison table image, training curves, metric bars, accuracy–efficiency frontier, sample GT masks, lesion-area distribution